# Credit Scoring Model
**Objective:** Predict an individual's creditworthiness from past financial data.

**Models:** Logistic Regression, Decision Tree, Random Forest  
**Metrics:** Precision, Recall, F1-Score, ROC-AUC

Run the cells in order (Runtime → Run all). All libraries are pre-installed in Colab.

## 1. Write the project code

In [ ]:
%%writefile credit_scoring.py
"""
Credit Scoring Model
====================
Predict an individual's creditworthiness from past financial data.

Models : Logistic Regression, Decision Tree, Random Forest
Metrics: Precision, Recall, F1-Score, ROC-AUC

Usage
-----
    python credit_scoring.py                      # uses synthetic data
    python credit_scoring.py --data my_data.csv   # uses your own CSV
                                                  # (needs a 'creditworthy' column: 1 = good, 0 = bad)
"""
import argparse
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
TARGET = "creditworthy"  # 1 = creditworthy (good), 0 = not creditworthy (bad)
OUT = Path("outputs")


# --------------------------------------------------------------------------- #
# 1. Data
# --------------------------------------------------------------------------- #
def generate_synthetic_data(n: int = 6000, seed: int = SEED) -> pd.DataFrame:
    """Create a realistic synthetic credit dataset (income, debts, payment history...)."""
    rng = np.random.default_rng(seed)

    age = rng.integers(21, 70, n)
    income = rng.lognormal(mean=10.8, sigma=0.5, size=n).round(0)            # annual
    employment_years = np.clip(rng.normal((age - 21) * 0.5, 3), 0, None).round(1)
    credit_history_years = np.clip(rng.normal((age - 20) * 0.6, 3), 0.5, None).round(1)
    num_credit_lines = rng.poisson(4, n) + 1
    total_debt = (income * rng.beta(2, 4, n) * rng.uniform(0.3, 2.0, n)).round(0)
    loan_amount = (income * rng.uniform(0.1, 1.2, n)).round(0)
    credit_utilization = np.clip(rng.beta(2, 3, n), 0, 1).round(3)
    late_30d = rng.poisson(0.6, n)
    late_90d = rng.poisson(0.15, n)
    inquiries_6m = rng.poisson(1.2, n)
    has_mortgage = rng.binomial(1, 0.35, n)

    # latent risk -> probability of default
    z = (-1.2
         + 2.2 * (total_debt / income)
         + 1.0 * (loan_amount / income)
         + 1.8 * credit_utilization
         + 0.45 * late_30d + 1.1 * late_90d
         + 0.20 * inquiries_6m
         - 0.06 * credit_history_years
         - 0.04 * employment_years
         - 0.25 * has_mortgage
         + rng.normal(0, 0.6, n))
    p_default = 1 / (1 + np.exp(-(z - 1.0)))
    default = rng.binomial(1, p_default)

    df = pd.DataFrame({
        "age": age, "income": income, "employment_years": employment_years,
        "credit_history_years": credit_history_years, "num_credit_lines": num_credit_lines,
        "total_debt": total_debt, "loan_amount": loan_amount,
        "credit_utilization": credit_utilization, "late_payments_30d": late_30d,
        "late_payments_90d": late_90d, "credit_inquiries_6m": inquiries_6m,
        "has_mortgage": has_mortgage, TARGET: 1 - default,
    })

    # inject missing values like real-world data
    for col in ["income", "employment_years", "credit_utilization"]:
        df.loc[rng.random(n) < 0.03, col] = np.nan
    return df


# --------------------------------------------------------------------------- #
# 2. Feature engineering
# --------------------------------------------------------------------------- #
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Derive ratios and summary features from raw financial history."""
    df = df.copy()
    income = df["income"].replace(0, np.nan)

    df["debt_to_income"] = df["total_debt"] / income
    df["loan_to_income"] = df["loan_amount"] / income
    df["log_income"] = np.log1p(df["income"])
    df["total_late_payments"] = df["late_payments_30d"] + df["late_payments_90d"]
    # weight severe delinquencies more heavily
    df["payment_history_score"] = 1 / (1 + df["late_payments_30d"] + 3 * df["late_payments_90d"])
    df["avg_account_age"] = df["credit_history_years"] / df["num_credit_lines"]
    df["high_utilization_flag"] = (df["credit_utilization"] > 0.7).astype(int)
    df["debt_per_credit_line"] = df["total_debt"] / df["num_credit_lines"]
    return df


# --------------------------------------------------------------------------- #
# 3. Models
# --------------------------------------------------------------------------- #
def build_models() -> dict:
    def pipe(clf, scale=False):
        steps = [("impute", SimpleImputer(strategy="median"))]
        if scale:
            steps.append(("scale", StandardScaler()))
        steps.append(("clf", clf))
        return Pipeline(steps)

    return {
        "Logistic Regression": pipe(
            LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED), scale=True),
        "Decision Tree": pipe(
            DecisionTreeClassifier(max_depth=6, min_samples_leaf=30,
                                   class_weight="balanced", random_state=SEED)),
        "Random Forest": pipe(
            RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=10,
                                   class_weight="balanced", n_jobs=-1, random_state=SEED)),
    }


# --------------------------------------------------------------------------- #
# 4. Evaluation
# --------------------------------------------------------------------------- #
def evaluate(models, X_train, X_test, y_train, y_test):
    rows, probas = [], {}
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        probas[name] = proba
        cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()
        rows.append({
            "Model": name,
            "Precision": precision_score(y_test, pred),
            "Recall": recall_score(y_test, pred),
            "F1-Score": f1_score(y_test, pred),
            "ROC-AUC": roc_auc_score(y_test, proba),
            "CV ROC-AUC (5-fold)": cv_auc,
        })
    return pd.DataFrame(rows).set_index("Model").round(4), probas


def plot_roc(probas, y_test, path):
    plt.figure(figsize=(7, 6))
    for name, p in probas.items():
        fpr, tpr, _ = roc_curve(y_test, p)
        plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(y_test, p):.3f})")
    plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curves - Credit Scoring Models")
    plt.legend(loc="lower right"); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(path, dpi=150); plt.close()


def plot_confusion(model, X_test, y_test, name, path):
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, display_labels=["Not creditworthy", "Creditworthy"],
        cmap="Blues", ax=ax)
    ax.set_title(f"Confusion Matrix - {name}")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def plot_importance(model, feature_names, path, top=12):
    imp = pd.Series(model.named_steps["clf"].feature_importances_, index=feature_names)
    imp = imp.sort_values().tail(top)
    plt.figure(figsize=(7, 5))
    imp.plot.barh(color="#3b82f6")
    plt.title("Random Forest - Feature Importance"); plt.xlabel("Importance")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


# --------------------------------------------------------------------------- #
# 5. Main
# --------------------------------------------------------------------------- #
def main(data_path=None):
    OUT.mkdir(exist_ok=True)

    df = pd.read_csv(data_path) if data_path else generate_synthetic_data()
    if not data_path:
        df.to_csv(OUT / "synthetic_credit_data.csv", index=False)
    print(f"Dataset: {df.shape[0]:,} rows | creditworthy rate = {df[TARGET].mean():.1%}")

    df = engineer_features(df)
    X, y = df.drop(columns=[TARGET]), df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED)

    models = build_models()
    results, probas = evaluate(models, X_train, X_test, y_train, y_test)
    print("\n=== Test-set results ===")
    print(results.to_string())
    results.to_csv(OUT / "model_comparison.csv")

    best = results["ROC-AUC"].idxmax()
    print(f"\nBest model by ROC-AUC: {best}")

    plot_roc(probas, y_test, OUT / "roc_curves.png")
    plot_confusion(models[best], X_test, y_test, best, OUT / "confusion_matrix.png")
    plot_importance(models["Random Forest"], X.columns, OUT / "feature_importance.png")
    joblib.dump({"model": models[best], "features": list(X.columns)}, OUT / "best_model.joblib")
    print(f"Artifacts saved to ./{OUT}/")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", help="Path to CSV with a 'creditworthy' target column")
    main(ap.parse_args().data)


## 2. Train and evaluate (uses synthetic data)

In [ ]:
!python credit_scoring.py

## 3. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv('outputs/model_comparison.csv', index_col=0))
for f in ['roc_curves', 'confusion_matrix', 'feature_importance']:
    display(Image(f'outputs/{f}.png'))

## 4. Score a new applicant

In [ ]:
%%writefile predict.py
"""Score a new applicant with the trained model."""
import joblib
import pandas as pd
from credit_scoring import engineer_features

bundle = joblib.load("outputs/best_model.joblib")

applicant = pd.DataFrame([{
    "age": 35, "income": 65000, "employment_years": 8, "credit_history_years": 10,
    "num_credit_lines": 5, "total_debt": 18000, "loan_amount": 20000,
    "credit_utilization": 0.35, "late_payments_30d": 0, "late_payments_90d": 0,
    "credit_inquiries_6m": 1, "has_mortgage": 1,
}])

X = engineer_features(applicant)[bundle["features"]]
p = bundle["model"].predict_proba(X)[0, 1]
print(f"Probability creditworthy: {p:.1%} -> {'APPROVE' if p >= 0.5 else 'REVIEW / DECLINE'}")


In [ ]:
!python predict.py

## 5. (Optional) Use your own data
Upload a CSV with a `creditworthy` column (1 = good, 0 = bad) and the same feature columns, then run the cell below.

In [ ]:
from google.colab import files
up = files.upload()
csv_name = next(iter(up))
!python credit_scoring.py --data "{csv_name}"